In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import uuid
uuid.uuid4()

In [0]:
import requests

# Your Census API key (optional but recommended)
API_KEY = "b3da786285a2b1ef4d04c61bd6d2ba304c8ecb84"

# url = "https://api.census.gov/data/2022/acs/acs1"

# params = {
#     "get": "NAME,B01003_001E",  # NAME = county name, B01003_001E = total population
#     "for": "county:*",           # all counties
#     "key": API_KEY
# }

# response = requests.get(url, params=params)
# try:
#     data = response.json()
# except Exception as e:
#     print(e)
#     raise e

# import pandas as pd

# # Convert to something usable
# try:
#     headers = data[0]
#     rows = data[1:]
#     df_counties = pd.DataFrame(rows, columns=headers)
#     df_counties["api_endpoint"] = url
#     df_counties["api_params"] = str(params)
#     display(df_counties.head())
# except Exception as e:
#     print(e)

In [0]:

url = "https://api.census.gov/data/2023/acs/acs5"

column_list = [
        "NAME",

        # Total population
        "B01003_001E",

        # Median age
        "B01002_001E",

        # Median household income
        "B19013_001E",

        # Poverty count
        "B17001_002E",

        # Race counts
        "B02001_002E",  # White alone
        "B02001_003E",  # Black alone
        "B02001_005E",  # Asian alone

        # Hispanic population
        "B03003_003E"
    ]

Zip Code

In [0]:
from pyspark.sql.functions import lit

params = {
    "get": ",".join(column_list),
    "for": "zip code tabulation area:*",
    "key": API_KEY
}

response = requests.get(url, params=params)
data = response.json()

# Extract header and rows
columns = data[0]
rows = data[1:]

# Create Spark DataFrame directly
df = spark.createDataFrame(rows, schema=columns)

df = df.withColumn("source", lit("census_api")) \
       .withColumn("api_endpoint", lit(url)) \
       .withColumn("api_params", lit(
           str({k: v for k, v in params.items() if k != "key"})))
    
    
display(df)

In [0]:
import src.utils.helpers

df_prepped = src.utils.helpers.prep_bronze_api_df(df)

In [0]:
src.utils.helpers.upsert_table(
    table_name="bronze_dev.census_bureau.acs_zipcode", 
    df=df_prepped, 
    natural_key=["NAME"], 
    spark=spark)

In [0]:
%sql
SELECT * FROM bronze_dev.census_bureau.acs_zipcode